# GreekKV sweep — Colab runner

Runs the GreekKV T17+T20 sweep on a Colab session (CPU or T4). The notebook is a **thin wrapper** — all logic lives in `scripts/run_sweep.py` and `src/greekkv/eval/perplexity.py`. Edit `configs/sweep_*.yaml` to change the grid; the same JSONL output is readable by the T22 report regardless of which host ran it.

## 1 — Install dependencies

In [ ]:
# Colab fresh clone + editable install. On a free T4 runtime add
# `bitsandbytes` for int4 model loading.
!git clone https://github.com/georgebon5/greekkv.git --depth 1 || true
%cd greekkv
!pip install -q -e .
# OPTIONAL (T4 required):
# !pip install -q bitsandbytes

## 2 — Run the T21 sweep (resumable)

The sweep YAML lists every `(scheme, n_ctx)` cell. If the output JSONL already has rows for some of them (e.g. from a previous run), they are skipped by `config_hash` — only the missing cells are computed.

In [ ]:
!python scripts/run_sweep.py --config configs/sweep_perplexity_qwen_0_5b.yaml

## 3 — Inspect the output

The JSONL schema follows `greekkv.utils.ResultRecord` (T03), so the T22 report can regenerate tables + figures from this file without a re-run.

In [ ]:
import json

rows = []
with open("results/perplexity/qwen2_5_0_5b.jsonl") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))
print(f"{len(rows)} rows")
for r in rows[:3]:
    print(r["scheme"], r["metrics"]["n_ctx"], r["metrics"]["perplexity"])

## Tips

- Mount Drive and point `output:` to a path there to keep the JSONL across session restarts.
- Idle timeouts can kill a run mid-sweep — not a problem: re-run the sweep cell and the runner resumes from the last completed cell.